# 선호 미반영 SFCA · 선호 반영 H3SFCA · 문화시설 다양성

- 목적: 서울 결과 격자를 기준으로 하되, 외부 25km 경쟁수요와 경쟁공급을 반영한 문화시설 접근성 지표를 산출함.
- 순서: 선호 미반영 접근성 → 선호 반영 접근성 → 문화시설 다양성.
- 네트워크: 차도망이 아니라 도보 750m 또는 대중교통 20분 이내 접근 pair를 사용함.

## 1. 분석 범위와 사용 데이터

- 서울 100m 격자는 최종 결과 산출 대상임.
- 경기·인천 외부 25km 격자는 경쟁수요 계산에만 사용함.
- 서울 + 외부 25km 가맹점은 경쟁공급 계산에 사용함.
- 공급량은 팀원 공급량 산정표의 sqrt 보정값을 사용함.
- 외부 가맹점 공급량 결측은 서울 동일 소분류/중분류 중앙값으로 대체함.

## 2. 선호 미반영 SFCA 수식

- 모든 중분류의 선호확률을 1로 둠.
- 격자 총 추정인구를 동일한 경쟁수요로 사용함.

$$
R_{jc}=\frac{S_{jc}}{\sum_i P_i W(c_{ij})}
$$

$$
A_{ic}=\sum_j R_{jc}W(c_{ij})
$$

## 3. 선호 반영 H3SFCA 수식

- 문화누리대상자와 비문화누리대상자를 분리함.
- ML 선호확률이 있는 중분류는 성별·연령·장애여부별 확률을 적용함.
- ML 선호확률이 없는 음악·체육용품은 선호 미반영 SFCA 접근성으로 대체함.

$$
D_{ic}^{(\lambda)}
=\sum_g N^{gen}_{ig}p_{gc}
+\lambda\sum_g N^{mnc}_{ig}p_{gc}
$$

$$
H_{ijc}
=
\frac{S_{jc}W(c_{ij})}
{\sum_{k\in J(i,c)}S_{kc}W(c_{ik})}
$$

$$
R_{jc}^{(\lambda)}
=
\frac{S_{jc}}
{\sum_i D_{ic}^{(\lambda)}H_{ijc}}
$$

$$
A_{ic}^{(\lambda)}
=
\sum_j H_{ijc}R_{jc}^{(\lambda)}
$$


## 4. 거리감쇠·수요가중치 시나리오

- 선호 미반영 SFCA는 팀원 구간형 감쇠만 계산함.
- 선호 반영 H3SFCA는 팀원 구간형, 가우시안형, 감쇠 없음 3개를 계산함.
- 문화누리대상자 수요가중치는 1.0, 1.2, 1.5를 계산함.
- 기본 표시값은 팀원 구간형 감쇠 + 수요가중치 1.2로 저장함.

## 5. 실행

- 입력 데이터 품질을 확인함.
- 공급량 결측, 선호수요 구성, 접근 pair 수를 출력함.
- 주요 산출물은 `notebooks/access/OUTPUT/h3sfca`, 다양성 산출물은 `notebooks/access/OUTPUT/diversity`에 저장함.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyogrio

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.6f}".format)


BASE_PATH = Path().resolve()

if BASE_PATH.name == "access":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "analysis_table").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

ANALYSIS_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output"
NETWORK_OUTPUT_PATH = ANALYSIS_OUTPUT_PATH / "network_competition_25km"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "h3sfca"
DIVERSITY_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT" / "diversity"
DOCS_PATH = PROJECT_PATH / "notebooks" / "access" / "docs"
GRID_RAW_PATH = PROJECT_PATH / "data" / "grid" / "격자100m_성연령별인구_2024_10"
PREFERENCE_PATH = PROJECT_PATH / "notebooks" / "preference" / "ML" / "yunseon_pl_model_handoff" / "preference_lookup_seoul.csv"

ACCESS_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DIVERSITY_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DOCS_PATH.mkdir(parents=True, exist_ok=True)

GRID_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet"
STORE_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet"
WALK_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
TRANSIT_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"
SUPPLY_PATH = ACCESS_OUTPUT_PATH / "문화누리_가맹점_카테고리별_공급량.csv"
SEOUL_MNC_GRID_PATH = ANALYSIS_OUTPUT_PATH / "서울시_100m_문화누리추정인구수.gpkg"
EXTERNAL_MNC_GRID_PATH = ANALYSIS_OUTPUT_PATH / "인천경기_외부25km_100m_문화누리대상자_추정인구.gpkg"
SEOUL_MNC_AGE_DISABLED_PATH = ANALYSIS_OUTPUT_PATH / "서울시_격자_100m_문화누리대상자_성연령장애별_인구수.csv"
EXTERNAL_AGE_PATH = ANALYSIS_OUTPUT_PATH / "인천경기_외부25km_100m_성연령별_추정인구.parquet"
DISABLED_SIGUNGU_PATH = PROJECT_PATH / "data" / "grid" / "시군구별_장애정도별_성별_등록장애인수_20260814052825.csv"
RESIDENT_POP_PATH = PROJECT_PATH / "data" / "grid" / "202410_202410_주민등록인구및세대현황_월간.csv"

WALK_CATEGORIES = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
TRANSIT_CATEGORIES = ["미술", "공연", "스포츠관람", "관광지"]
TARGET_CATEGORIES = WALK_CATEGORIES + TRANSIT_CATEGORIES
CATEGORY_MODE = {**{category: "도보" for category in WALK_CATEGORIES}, **{category: "대중교통" for category in TRANSIT_CATEGORIES}}
ML_CATEGORIES = ["도서", "공연", "미술", "문화체험", "관광지", "스포츠관람", "체육시설", "영상"]
NON_ML_CATEGORIES = ["음악", "체육용품"]

AGE_RAW_CATEGORIES = [
    "총인구", "유아인구", "유소년인구",
    "20대인구", "30대인구", "40대인구", "50대인구",
    "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"
]
AGE_ALL_COLS = [
    "0-5세", "6-14세", "15-19세", "20-29세", "30-39세", "40-49세",
    "50-59세", "60-69세", "70-79세", "80-89세", "90-99세", "100세-"
]
AGE_MODEL_COLS = [
    "6-14세", "15-19세", "20-29세", "30-39세", "40-49세", "50-59세",
    "60-69세", "70-79세", "80-89세", "90-99세", "100세-"
]
AGE_CODE_MAP = {
    "6-14세": 1, "15-19세": 1,
    "20-29세": 2, "30-39세": 3, "40-49세": 4,
    "50-59세": 5, "60-69세": 6,
    "70-79세": 7, "80-89세": 7, "90-99세": 7, "100세-": 7,
}
SEX_CODE_MAP = {"남성": 1, "여성": 2}

WALK_CUTOFF_M = 750.0
TRANSIT_CUTOFF_MIN = 20.0
PRIMARY_LAMBDA = 1.2
LAMBDA_VALUES = [1.0, 1.2, 1.5]
DECAY_METHODS = ["piecewise", "gaussian", "none"]


def log(message):
    print(f"[H3SFCA] {message}")


def pick_col(df, include_words, exclude_words=None):
    exclude_words = exclude_words or []
    for col in df.columns:
        name = str(col)
        if all(word in name for word in include_words) and not any(word in name for word in exclude_words):
            return col
    raise KeyError(f"칼럼을 찾지 못했습니다: {include_words}")


def clean_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def normalize_age(value):
    value = str(value)
    if value == "100세이상":
        return "100세-"
    return value


def load_mnc_grid_counts():
    frames = []
    for path in [SEOUL_MNC_GRID_PATH, EXTERNAL_MNC_GRID_PATH]:
        temp = pyogrio.read_dataframe(path, read_geometry=False)
        grid_col = pick_col(temp, ["GRID"])
        mnc_col = pick_col(temp, ["문화누리", "추정"], exclude_words=["성연령", "장애"])
        temp = temp[[grid_col, mnc_col]].rename(columns={grid_col: "GRID_CD", mnc_col: "문화누리대상자_추정_인구수"})
        frames.append(temp)
    result = pd.concat(frames, ignore_index=True)
    result["문화누리대상자_추정_인구수"] = pd.to_numeric(result["문화누리대상자_추정_인구수"], errors="coerce").fillna(0)
    result = result.groupby("GRID_CD", as_index=False)["문화누리대상자_추정_인구수"].sum()
    return result


def load_grid_base():
    grid = pd.read_parquet(GRID_COMPETITION_PATH)
    grid = grid.drop(columns=["geometry"], errors="ignore").copy()
    mnc = load_mnc_grid_counts()
    grid = grid.merge(mnc, on="GRID_CD", how="left")
    grid["문화누리대상자_추정_인구수"] = pd.to_numeric(grid["문화누리대상자_추정_인구수"], errors="coerce").fillna(0).round().astype(int)
    grid["추정_인구수"] = pd.to_numeric(grid["추정_인구수"], errors="coerce").fillna(0).round().astype(int)
    grid["비문화누리대상자_추정_인구수"] = (grid["추정_인구수"] - grid["문화누리대상자_추정_인구수"]).clip(lower=0).astype(int)
    grid["서울여부"] = grid["서울여부"].astype(bool)
    log(f"분석 격자: {grid.shape}, 서울 {grid['서울여부'].sum():,}, 외부 {(~grid['서울여부']).sum():,}")
    log(f"총인구 합 {grid['추정_인구수'].sum():,}, 문화누리대상자 합 {grid['문화누리대상자_추정_인구수'].sum():,}")
    return grid


def load_store_supply():
    store = pd.read_parquet(STORE_COMPETITION_PATH)
    store = store.drop(columns=["geometry"], errors="ignore").copy()
    supply = pd.read_csv(SUPPLY_PATH, encoding="utf-8-sig")
    supply = supply[supply["분석포함"].astype(bool)].copy()
    supply["공급량"] = pd.to_numeric(supply["공급량"], errors="coerce").fillna(1).clip(lower=0)
    store = store.merge(
        supply[["가맹점_ID", "중분류", "소분류", "공급량", "공급량_raw", "공급기준"]],
        on=["가맹점_ID", "중분류", "소분류"],
        how="left",
    )
    sub_ref = supply.groupby(["중분류", "소분류"], as_index=False)["공급량"].median().rename(columns={"공급량": "소분류_보정공급량"})
    cat_ref = supply.groupby("중분류", as_index=False)["공급량"].median().rename(columns={"공급량": "중분류_보정공급량"})
    store = store.merge(sub_ref, on=["중분류", "소분류"], how="left")
    store = store.merge(cat_ref, on="중분류", how="left")
    missing_before = int(store["공급량"].isna().sum())
    store["공급량"] = store["공급량"].fillna(store["소분류_보정공급량"]).fillna(store["중분류_보정공급량"]).fillna(1)
    store["공급량_raw"] = pd.to_numeric(store["공급량_raw"], errors="coerce").fillna(store["공급량"] ** 2)
    store["공급기준"] = store["공급기준"].fillna("서울 동일 소분류/중분류 공급량 중앙값 대체")
    store = store[store["중분류"].isin(TARGET_CATEGORIES)].copy()
    store["공급량"] = store["공급량"].clip(lower=0)
    log(f"공급량 적용 가맹점: {store.shape}, 공급량 대체 {missing_before:,}행")
    display(
        store.groupby(["서울여부", "중분류"], as_index=False)
        .agg(가맹점수=("가맹점_ID", "nunique"), 보정공급량합=("공급량", "sum"))
        .head(30)
    )
    return store


def parse_sigungu_population():
    pop = pd.read_csv(RESIDENT_POP_PATH, encoding="cp949")
    pop["총인구수"] = clean_number(pop["2024년10월_총인구수"]).fillna(0)
    parsed = pop["행정구역"].astype(str).str.extract(r"^(?P<행정명>.+?)\s*\((?P<행정코드>\d+)\)$")
    pop["행정명"] = parsed["행정명"].str.strip()
    pop["행정코드"] = parsed["행정코드"]
    sigungu = pop[pop["행정코드"].str.endswith("00000", na=False)].copy()
    sigungu = sigungu[~sigungu["행정코드"].str.endswith("00000000", na=False)].copy()
    parts = sigungu["행정명"].str.split()
    sigungu["시도"] = parts.str[0]
    sigungu["시군구"] = parts.apply(lambda x: " ".join(x[1:]) if isinstance(x, list) and len(x) > 1 else np.nan)
    return sigungu[["시도", "시군구", "총인구수"]]


def parse_disabled_ratio():
    disabled = pd.read_csv(DISABLED_SIGUNGU_PATH, encoding="utf-8-sig")
    disabled = disabled.iloc[2:].copy()
    disabled = disabled.rename(columns={disabled.columns[0]: "시도", disabled.columns[1]: "시군구", disabled.columns[2]: "등록장애인수"})
    disabled["등록장애인수"] = clean_number(disabled["등록장애인수"]).fillna(0)
    disabled = disabled[disabled["시도"].isin(["서울특별시", "경기도", "인천광역시"])].copy()
    disabled = disabled[disabled["시군구"].ne("소계")].copy()
    pop = parse_sigungu_population()
    ratio = disabled[["시도", "시군구", "등록장애인수"]].merge(pop, on=["시도", "시군구"], how="left")
    ratio["장애인비율"] = np.where(ratio["총인구수"] > 0, ratio["등록장애인수"] / ratio["총인구수"], np.nan)
    sido_ratio = (
        ratio.groupby("시도", as_index=False)
        .agg(등록장애인수=("등록장애인수", "sum"), 총인구수=("총인구수", "sum"))
    )
    sido_ratio["시도_장애인비율"] = np.where(sido_ratio["총인구수"] > 0, sido_ratio["등록장애인수"] / sido_ratio["총인구수"], 0)
    ratio = ratio.merge(sido_ratio[["시도", "시도_장애인비율"]], on="시도", how="left")
    ratio["장애인비율"] = ratio["장애인비율"].fillna(ratio["시도_장애인비율"]).fillna(0).clip(0, 1)
    log(f"장애인비율: {ratio.shape}, 결측 {ratio['장애인비율'].isna().sum():,}")
    return ratio[["시도", "시군구", "장애인비율"]]


def read_seoul_age_raw(grid_ids):
    grid_ids = set(grid_ids)
    frames = []
    for age in AGE_RAW_CATEGORIES:
        for sex in ["남성", "여성"]:
            path = GRID_RAW_PATH / age / sex
            if not path.exists():
                continue
            for shp_path in path.rglob("vl_blk.shp"):
                temp = pyogrio.read_dataframe(
                    shp_path,
                    columns=["gid", "val"],
                    read_geometry=False,
                    encoding="utf-8",
                )
                temp = temp.rename(columns={"gid": "GRID_CD", "val": "인구수"})
                temp = temp[temp["GRID_CD"].isin(grid_ids)].copy()
                if len(temp) == 0:
                    continue
                temp["성별"] = sex
                temp["연령대원자료"] = age
                frames.append(temp)
    raw = pd.concat(frames, ignore_index=True)
    raw["인구수"] = pd.to_numeric(raw["인구수"], errors="coerce").fillna(0)
    raw = raw.groupby(["GRID_CD", "성별", "연령대원자료"], as_index=False)["인구수"].sum()
    log(f"서울 성연령 원자료: {raw.shape}, GRID {raw['GRID_CD'].nunique():,}")
    return raw


def build_age_ratio_from_raw(grid, raw):
    wide = raw.pivot_table(
        index=["GRID_CD", "성별"],
        columns="연령대원자료",
        values="인구수",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
    wide.columns.name = None
    for col in AGE_RAW_CATEGORIES:
        if col not in wide.columns:
            wide[col] = 0
    over20 = ["20대인구", "30대인구", "40대인구", "50대인구", "60대인구", "70대인구", "80대인구", "90대인구", "100세이상인구"]
    wide["0-19세"] = wide["총인구"] - wide[over20].sum(axis=1)
    wide["0-5세"] = wide["유아인구"]
    wide["6-14세"] = wide["유소년인구"] - wide["0-5세"]
    wide["15-19세"] = wide["0-19세"] - wide["유소년인구"]
    wide["20-29세"] = wide["20대인구"]
    wide["30-39세"] = wide["30대인구"]
    wide["40-49세"] = wide["40대인구"]
    wide["50-59세"] = wide["50대인구"]
    wide["60-69세"] = wide["60대인구"]
    wide["70-79세"] = wide["70대인구"]
    wide["80-89세"] = wide["80대인구"]
    wide["90-99세"] = wide["90대인구"]
    wide["100세-"] = wide["100세이상인구"]
    negative = {col: int((wide[col] < 0).sum()) for col in ["0-19세", "6-14세", "15-19세"]}
    for col in AGE_ALL_COLS:
        wide[col] = pd.to_numeric(wide[col], errors="coerce").fillna(0).clip(lower=0)
    long = wide.melt(id_vars=["GRID_CD", "성별"], value_vars=AGE_ALL_COLS, var_name="연령대", value_name="원자료_인구수")
    sex_age = pd.MultiIndex.from_product([["남성", "여성"], AGE_ALL_COLS], names=["성별", "연령대"]).to_frame(index=False)
    full = grid[["GRID_CD", "시군구", "행정동", "비문화누리대상자_추정_인구수"]].merge(sex_age, how="cross")
    full = full.merge(long, on=["GRID_CD", "성별", "연령대"], how="left")
    full["원자료_인구수"] = full["원자료_인구수"].fillna(0)
    full["격자_원자료합"] = full.groupby("GRID_CD")["원자료_인구수"].transform("sum")
    full["격자_비중"] = np.where(full["격자_원자료합"] > 0, full["원자료_인구수"] / full["격자_원자료합"], np.nan)
    dong = full.groupby(["시군구", "행정동", "성별", "연령대"], as_index=False)["원자료_인구수"].sum()
    dong["행정동_원자료합"] = dong.groupby(["시군구", "행정동"])["원자료_인구수"].transform("sum")
    dong["행정동_비중"] = np.where(dong["행정동_원자료합"] > 0, dong["원자료_인구수"] / dong["행정동_원자료합"], 0)
    full = full.merge(dong[["시군구", "행정동", "성별", "연령대", "행정동_비중"]], on=["시군구", "행정동", "성별", "연령대"], how="left")
    full["최종_비중"] = np.where(full["격자_원자료합"] > 0, full["격자_비중"], full["행정동_비중"])
    full["최종_비중"] = full["최종_비중"].fillna(0)
    full["비문화누리대상자_성연령별_추정_인구수"] = full["비문화누리대상자_추정_인구수"] * full["최종_비중"]
    log(f"서울 성연령 파생 음수 후보: {negative}")
    return full


def build_seoul_demand_strata(grid, disabled_ratio):
    seoul_grid = grid[grid["서울여부"]].copy()
    raw = read_seoul_age_raw(seoul_grid["GRID_CD"])
    non_mnc = build_age_ratio_from_raw(seoul_grid, raw)
    grid_disabled_ratio = pd.read_csv(
        SEOUL_MNC_AGE_DISABLED_PATH,
        usecols=["GRID_CD", "행정동_장애인비율"],
        encoding="utf-8-sig",
    ).drop_duplicates("GRID_CD")
    non_mnc = non_mnc.merge(grid_disabled_ratio, on="GRID_CD", how="left")
    non_mnc["행정동_장애인비율"] = pd.to_numeric(non_mnc["행정동_장애인비율"], errors="coerce").fillna(0).clip(0, 1)
    non_mnc = non_mnc[non_mnc["연령대"].isin(AGE_MODEL_COLS)].copy()
    non_disabled = non_mnc.copy()
    non_disabled["장애여부코드"] = 3
    non_disabled["일반인구"] = non_disabled["비문화누리대상자_성연령별_추정_인구수"] * (1 - non_disabled["행정동_장애인비율"])
    disabled = non_mnc.copy()
    disabled["장애여부코드"] = 1
    disabled["일반인구"] = disabled["비문화누리대상자_성연령별_추정_인구수"] * disabled["행정동_장애인비율"]
    general = pd.concat([non_disabled, disabled], ignore_index=True)
    general = general[["GRID_CD", "성별", "연령대", "장애여부코드", "일반인구"]].copy()

    mnc = pd.read_csv(
        SEOUL_MNC_AGE_DISABLED_PATH,
        usecols=[
            "GRID_CD", "성별", "연령대",
            "문화누리대상자_성연령장애별_추정_인구수",
            "문화누리대상자_성연령비장애별_추정_인구수",
        ],
        encoding="utf-8-sig",
    )
    mnc = mnc[mnc["연령대"].isin(AGE_MODEL_COLS)].copy()
    mnc_dis = mnc[["GRID_CD", "성별", "연령대", "문화누리대상자_성연령장애별_추정_인구수"]].rename(columns={"문화누리대상자_성연령장애별_추정_인구수": "문화누리대상자"})
    mnc_dis["장애여부코드"] = 1
    mnc_non = mnc[["GRID_CD", "성별", "연령대", "문화누리대상자_성연령비장애별_추정_인구수"]].rename(columns={"문화누리대상자_성연령비장애별_추정_인구수": "문화누리대상자"})
    mnc_non["장애여부코드"] = 3
    mnc = pd.concat([mnc_dis, mnc_non], ignore_index=True)
    result = general.merge(mnc, on=["GRID_CD", "성별", "연령대", "장애여부코드"], how="outer")
    result[["일반인구", "문화누리대상자"]] = result[["일반인구", "문화누리대상자"]].fillna(0)
    result["성별코드"] = result["성별"].map(SEX_CODE_MAP)
    result["연령코드"] = result["연령대"].map(AGE_CODE_MAP)
    return result


def build_external_demand_strata(disabled_ratio):
    external = pd.read_parquet(EXTERNAL_AGE_PATH)
    external["연령대"] = external["연령대"].map(normalize_age)
    external = external[external["연령대"].isin(AGE_MODEL_COLS)].copy()
    external = external.merge(disabled_ratio, on=["시도", "시군구"], how="left")
    sido_ratio = disabled_ratio.groupby("시도", as_index=False)["장애인비율"].mean().rename(columns={"장애인비율": "시도_장애인비율"})
    external = external.merge(sido_ratio, on="시도", how="left")
    external["장애인비율"] = external["장애인비율"].fillna(external["시도_장애인비율"]).fillna(0).clip(0, 1)
    frames = []
    for code, ratio_expr in [(1, external["장애인비율"]), (3, 1 - external["장애인비율"])]:
        temp = external.copy()
        temp["장애여부코드"] = code
        temp["일반인구"] = temp["비문화누리대상자_성연령별_추정_인구수"] * ratio_expr
        temp["문화누리대상자"] = temp["문화누리대상자_성연령별_추정_인구수"] * ratio_expr
        frames.append(temp[["GRID_CD", "성별", "연령대", "장애여부코드", "일반인구", "문화누리대상자"]])
    result = pd.concat(frames, ignore_index=True)
    result["성별코드"] = result["성별"].map(SEX_CODE_MAP)
    result["연령코드"] = result["연령대"].map(AGE_CODE_MAP)
    return result


def build_preference_lookup():
    pref = pd.read_csv(PREFERENCE_PATH, encoding="utf-8-sig")
    pref = pref.rename(columns={"관광": "관광지", "영화": "영상"})
    general_prob = (
        pref.groupby(["성별", "연령", "장애여부"], as_index=False)[ML_CATEGORIES]
        .mean()
        .melt(id_vars=["성별", "연령", "장애여부"], value_vars=ML_CATEGORIES, var_name="중분류", value_name="일반인구_선호확률")
    )
    mnc_prob = (
        pref[pref["가구소득"].isin([1, 2, 3])]
        .groupby(["성별", "연령", "장애여부"], as_index=False)[ML_CATEGORIES]
        .mean()
        .melt(id_vars=["성별", "연령", "장애여부"], value_vars=ML_CATEGORIES, var_name="중분류", value_name="문화누리대상자_선호확률")
    )
    return general_prob, mnc_prob


def build_preference_demand(grid, demand_strata):
    general_prob, mnc_prob = build_preference_lookup()
    general = demand_strata.merge(
        general_prob,
        left_on=["성별코드", "연령코드", "장애여부코드"],
        right_on=["성별", "연령", "장애여부"],
        how="left",
        suffixes=("", "_prob"),
    )
    general["일반인구_선호수요"] = general["일반인구"] * general["일반인구_선호확률"].fillna(0)
    general_demand = general.groupby(["GRID_CD", "중분류"], as_index=False)["일반인구_선호수요"].sum()
    mnc = demand_strata.merge(
        mnc_prob,
        left_on=["성별코드", "연령코드", "장애여부코드"],
        right_on=["성별", "연령", "장애여부"],
        how="left",
        suffixes=("", "_prob"),
    )
    mnc["문화누리대상자_기본선호수요"] = mnc["문화누리대상자"] * mnc["문화누리대상자_선호확률"].fillna(0)
    mnc_demand = mnc.groupby(["GRID_CD", "중분류"], as_index=False)["문화누리대상자_기본선호수요"].sum()
    base = grid[["GRID_CD"]].merge(pd.DataFrame({"중분류": ML_CATEGORIES}), how="cross")
    base = base.merge(general_demand, on=["GRID_CD", "중분류"], how="left")
    base = base.merge(mnc_demand, on=["GRID_CD", "중분류"], how="left")
    base[["일반인구_선호수요", "문화누리대상자_기본선호수요"]] = base[["일반인구_선호수요", "문화누리대상자_기본선호수요"]].fillna(0)
    demand = base.copy()
    log(f"선호수요 테이블: {demand.shape}")
    return demand


def load_access_pair(store):
    pair_frames = []
    path_mode_list = [
        (WALK_PAIR_PATH, "도보"),
        (TRANSIT_PAIR_PATH, "대중교통"),
    ]
    for path, mode in path_mode_list:
        temp = pd.read_parquet(path)
        temp["접근비용"] = pd.to_numeric(temp["접근비용"], errors="coerce").astype("float32")
        temp["접근수단"] = mode
        if mode == "도보":
            temp = temp[temp["접근비용"].between(0, WALK_CUTOFF_M)].copy()
        else:
            temp = temp[temp["접근비용"].between(0, TRANSIT_CUTOFF_MIN)].copy()
        temp = temp.merge(store[["가맹점_ID", "중분류", "공급량"]], on="가맹점_ID", how="left")
        temp = temp[temp["중분류"].isin(TARGET_CATEGORIES)].copy()
        temp = temp[temp["중분류"].map(CATEGORY_MODE).eq(temp["접근수단"])].copy()
        temp = temp[temp["공급량"].notna()].copy()
        temp["공급량"] = pd.to_numeric(temp["공급량"], errors="coerce").fillna(1).astype("float32")
        temp["중분류"] = temp["중분류"].astype("category")
        pair_frames.append(temp)
        log(f"{mode} 접근 pair: {temp.shape}, GRID {temp['GRID_CD'].nunique():,}, STORE {temp['가맹점_ID'].nunique():,}")
    pair = pd.concat(pair_frames, ignore_index=True)
    log(f"도보+대중교통 접근 pair: {pair.shape}, GRID {pair['GRID_CD'].nunique():,}, STORE {pair['가맹점_ID'].nunique():,}")
    return pair


def apply_decay(cost, mode, method):
    cost = pd.to_numeric(cost, errors="coerce")
    mode = pd.Series(mode, index=cost.index)
    weight = np.zeros(len(cost), dtype="float32")
    if method == "none":
        walk_idx = mode.eq("도보")
        transit_idx = mode.eq("대중교통")
        weight[walk_idx & (cost <= WALK_CUTOFF_M)] = 1.0
        weight[transit_idx & (cost <= TRANSIT_CUTOFF_MIN)] = 1.0
        return weight
    if method == "gaussian":
        walk_idx = mode.eq("도보")
        transit_idx = mode.eq("대중교통")
        sigma_walk = WALK_CUTOFF_M / math.sqrt(-2 * math.log(0.05))
        sigma_transit = TRANSIT_CUTOFF_MIN / math.sqrt(-2 * math.log(0.05))
        weight[walk_idx] = np.exp(-0.5 * (cost[walk_idx] / sigma_walk) ** 2)
        weight[transit_idx] = np.exp(-0.5 * (cost[transit_idx] / sigma_transit) ** 2)
        return weight.astype("float32")

    walk_idx = mode.eq("도보")
    transit_idx = mode.eq("대중교통")
    weight[walk_idx & (cost >= 0) & (cost <= 250)] = 1.0
    weight[walk_idx & (cost > 250) & (cost <= 500)] = 0.6
    weight[walk_idx & (cost > 500) & (cost <= 750)] = 0.25
    weight[transit_idx & (cost >= 0) & (cost <= 7)] = 1.0
    weight[transit_idx & (cost > 7) & (cost <= 14)] = 0.6
    weight[transit_idx & (cost > 14) & (cost <= 20)] = 0.25
    return weight



def seoul_grid_category_base(grid):
    return (
        grid[grid["서울여부"]][[
            "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
            "추정_인구수", "문화누리대상자_추정_인구수", "비문화누리대상자_추정_인구수"
        ]]
        .merge(pd.DataFrame({"중분류": TARGET_CATEGORIES}), how="cross")
    )


def run_no_preference_sfca(grid, pair):
    log("선호 미반영 SFCA 계산")
    demand = grid[["GRID_CD", "추정_인구수"]].rename(columns={"추정_인구수": "기준수요"})
    calc = pair[["GRID_CD", "가맹점_ID", "중분류", "접근수단", "접근비용", "공급량"]].copy()
    calc["거리감쇠"] = apply_decay(calc["접근비용"], calc["접근수단"], "piecewise")
    calc = calc[calc["거리감쇠"] > 0].copy()
    calc = calc.merge(demand, on="GRID_CD", how="left")
    calc["가중수요"] = calc["기준수요"].fillna(0) * calc["거리감쇠"]
    facility = calc.groupby(["가맹점_ID", "중분류"], observed=True, as_index=False).agg(
        공급량=("공급량", "first"),
        가중수요=("가중수요", "sum"),
    )
    facility["공급수요비"] = np.where(facility["가중수요"] > 0, facility["공급량"] / facility["가중수요"], 0)
    calc = calc.merge(facility[["가맹점_ID", "중분류", "공급수요비"]], on=["가맹점_ID", "중분류"], how="left")
    calc["접근성기여"] = calc["공급수요비"].fillna(0) * calc["거리감쇠"]
    seoul_ids = set(grid.loc[grid["서울여부"], "GRID_CD"])
    access = calc[calc["GRID_CD"].isin(seoul_ids)].groupby(["GRID_CD", "중분류"], observed=True, as_index=False).agg(
        접근성지수=("접근성기여", "sum"),
        접근가능_가맹점수=("가맹점_ID", "nunique"),
        평균접근비용=("접근비용", "mean"),
    )
    complete = seoul_grid_category_base(grid).merge(access, on=["GRID_CD", "중분류"], how="left")
    complete[["접근성지수", "접근가능_가맹점수", "평균접근비용"]] = complete[["접근성지수", "접근가능_가맹점수", "평균접근비용"]].fillna(0)
    complete["접근성_모형"] = "SFCA_선호미반영"
    complete["거리감쇠방식"] = "piecewise"
    complete["문화누리대상자_수요가중치"] = 1.0
    complete["수요량"] = complete["추정_인구수"]
    complete["선호수요"] = complete["수요량"]
    return complete, facility


def run_h3sfca_scenario(grid, pair, demand, decay_method, lambda_value):
    calc = pair[["GRID_CD", "가맹점_ID", "중분류", "접근수단", "접근비용", "공급량"]].copy()
    calc["거리감쇠"] = apply_decay(calc["접근비용"], calc["접근수단"], decay_method)
    calc = calc[calc["거리감쇠"] > 0].copy()
    calc["매력도"] = calc["공급량"] * calc["거리감쇠"]
    hden = calc.groupby(["GRID_CD", "중분류"], observed=True)["매력도"].transform("sum")
    calc["Huff확률"] = np.where(hden > 0, calc["매력도"] / hden, 0).astype("float32")
    demand_now = demand.copy()
    demand_now["선호수요"] = demand_now["일반인구_선호수요"] + lambda_value * demand_now["문화누리대상자_기본선호수요"]
    calc = calc.merge(demand_now[["GRID_CD", "중분류", "선호수요"]], on=["GRID_CD", "중분류"], how="left")
    calc["가중수요"] = calc["선호수요"].fillna(0) * calc["Huff확률"]
    facility = calc.groupby(["가맹점_ID", "중분류"], observed=True, as_index=False).agg(
        공급량=("공급량", "first"),
        가중수요=("가중수요", "sum"),
    )
    facility["공급수요비"] = np.where(facility["가중수요"] > 0, facility["공급량"] / facility["가중수요"], 0)
    calc = calc.merge(facility[["가맹점_ID", "중분류", "공급수요비"]], on=["가맹점_ID", "중분류"], how="left")
    calc["접근성기여"] = calc["공급수요비"].fillna(0) * calc["Huff확률"]
    seoul_ids = set(grid.loc[grid["서울여부"], "GRID_CD"])
    access = calc[calc["GRID_CD"].isin(seoul_ids)].groupby(["GRID_CD", "중분류"], observed=True, as_index=False).agg(
        접근성지수=("접근성기여", "sum"),
        접근가능_가맹점수=("가맹점_ID", "nunique"),
        평균접근비용=("접근비용", "mean"),
    )
    complete = seoul_grid_category_base(grid).merge(access, on=["GRID_CD", "중분류"], how="left")
    complete = complete.merge(demand_now[["GRID_CD", "중분류", "선호수요"]], on=["GRID_CD", "중분류"], how="left")
    complete[["접근성지수", "접근가능_가맹점수", "평균접근비용", "선호수요"]] = complete[["접근성지수", "접근가능_가맹점수", "평균접근비용", "선호수요"]].fillna(0)
    complete["접근성_모형"] = "H3SFCA_선호반영"
    complete["거리감쇠방식"] = decay_method
    complete["문화누리대상자_수요가중치"] = lambda_value
    complete["수요량"] = complete["선호수요"]
    facility["접근성_모형"] = "H3SFCA_선호반영"
    facility["거리감쇠방식"] = decay_method
    facility["문화누리대상자_수요가중치"] = lambda_value
    return complete, facility


def summarize_access(result):
    return result.groupby(["접근성_모형", "거리감쇠방식", "문화누리대상자_수요가중치", "중분류"], as_index=False).agg(
        격자수=("GRID_CD", "nunique"),
        수요합=("수요량", "sum"),
        접근성평균=("접근성지수", "mean"),
        접근성중앙값=("접근성지수", "median"),
        접근성최댓값=("접근성지수", "max"),
        무접근격자수=("접근가능_가맹점수", lambda x: int((x == 0).sum())),
        평균도달가맹점수=("접근가능_가맹점수", "mean"),
    )


def replace_non_ml_with_sfca(h3_result, sfca_result):
    replace_cols = ["접근성지수", "접근가능_가맹점수", "평균접근비용", "수요량", "선호수요"]
    key_cols = ["GRID_CD", "중분류"]
    sfca_replacement = sfca_result[sfca_result["중분류"].isin(NON_ML_CATEGORIES)][key_cols + replace_cols].copy()
    original_cols = [col for col in h3_result.columns if col != "선호확률_처리방식"]

    h3 = h3_result.copy()
    h3["선호확률_처리방식"] = np.where(
        h3["중분류"].isin(NON_ML_CATEGORIES),
        "선호미반영_SFCA대체",
        "ML선호확률적용",
    )

    ml_part = h3[~h3["중분류"].isin(NON_ML_CATEGORIES)].copy()
    non_ml_part = h3[h3["중분류"].isin(NON_ML_CATEGORIES)].drop(columns=replace_cols, errors="ignore")
    non_ml_part = non_ml_part.merge(sfca_replacement, on=key_cols, how="left")
    non_ml_part[replace_cols] = non_ml_part[replace_cols].fillna(0)

    result = pd.concat([ml_part, non_ml_part], ignore_index=True)
    return result[original_cols + ["선호확률_처리방식"]]


def replace_non_ml_facility_with_sfca(h3_facility, sfca_facility):
    replace_cols = ["공급량", "가중수요", "공급수요비"]
    key_cols = ["가맹점_ID", "중분류"]
    sfca_replacement = sfca_facility[sfca_facility["중분류"].isin(NON_ML_CATEGORIES)][key_cols + replace_cols].copy()
    original_cols = [col for col in h3_facility.columns if col != "선호확률_처리방식"]

    h3 = h3_facility.copy()
    h3["선호확률_처리방식"] = np.where(
        h3["중분류"].isin(NON_ML_CATEGORIES),
        "선호미반영_SFCA대체",
        "ML선호확률적용",
    )

    ml_part = h3[~h3["중분류"].isin(NON_ML_CATEGORIES)].copy()
    non_ml_part = h3[h3["중분류"].isin(NON_ML_CATEGORIES)].drop(columns=replace_cols, errors="ignore")
    non_ml_part = non_ml_part.merge(sfca_replacement, on=key_cols, how="left")
    non_ml_part[replace_cols] = non_ml_part[replace_cols].fillna(0)

    result = pd.concat([ml_part, non_ml_part], ignore_index=True)
    return result[original_cols + ["선호확률_처리방식"]]


def build_diversity(grid, pair):
    log("문화시설 다양성 지수 계산")
    seoul = grid[grid["서울여부"]][[
        "GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y",
        "추정_인구수", "문화누리대상자_추정_인구수"
    ]].copy()
    diversity = pair[pair["GRID_CD"].isin(set(seoul["GRID_CD"]))].groupby("GRID_CD", as_index=False).agg(
        diversity_type_n=("중분류", "nunique"),
        reachable_store_n=("가맹점_ID", "nunique"),
    )
    result = seoul.merge(diversity, on="GRID_CD", how="left")
    result[["diversity_type_n", "reachable_store_n"]] = result[["diversity_type_n", "reachable_store_n"]].fillna(0).astype(int)
    result["diversity_max_type_n"] = len(TARGET_CATEGORIES)
    result["diversity_ratio"] = result["diversity_type_n"] / len(TARGET_CATEGORIES)
    result["diversity_vulnerability"] = 1 - result["diversity_ratio"]
    out_path = DIVERSITY_OUTPUT_PATH / "grid_pop_access_diversity.csv"
    result.to_csv(out_path, index=False, encoding="utf-8-sig")
    log(f"다양성 산출물 저장: {out_path}")
    return result


def save_docs():
    sfca_doc = """# 선호 미반영 SFCA 접근성

## 사용 데이터
- 100m 격자 인구: 서울 + 외부 25km 경쟁권 추정인구
- 문화누리 가맹점: 서울 + 외부 25km 가맹점
- 접근 pair: 대중교통 네트워크 기준 20분 이내 격자-가맹점 pair
- 공급량: 팀원 공급량 산정표의 보정공급량(sqrt 변환값)

## 전처리/분석 방식
- 차도망은 사용하지 않고, 중분류별 도보 또는 대중교통 접근 pair를 사용함.
- 외부 가맹점 공급량은 서울 동일 소분류/중분류 보정공급량 중앙값으로 대체함.
- 선호확률은 사용하지 않고, 격자 총 추정인구를 중분류별 동일 수요로 적용함.
- 도보 거리감쇠는 0~250m 1.00, 250~500m 0.60, 500~750m 0.25를 적용하고, 대중교통 거리감쇠는 0~7분 1.00, 7~14분 0.60, 14~20분 0.25를 적용함.

## 주요 산출물
- sfca_no_preference_격자_중분류_접근성.csv
- sfca_no_preference_중분류_요약.csv
- sfca_no_preference_가맹점_공급수요비.csv
"""
    h3_doc = """# 선호 반영 H3SFCA 접근성

## 사용 데이터
- 100m 격자 인구: 서울 + 외부 25km 경쟁권 추정인구 및 문화누리대상자 추정인구
- 성연령 수요: 서울 성연령 원자료, 서울 문화누리 성연령장애 추정표, 외부 25km 성연령 추정표
- 장애여부: 서울 행정동 장애인비율, 경기·인천 시군구 등록장애인비율
- 선호확률: 팀원 ML 결과 preference_lookup_seoul.csv
- 공급량: 팀원 공급량 산정표의 보정공급량(sqrt 변환값)
- 접근 pair: 대중교통 네트워크 기준 20분 이내 격자-가맹점 pair

## 전처리/분석 방식
- 문화누리대상자와 비문화누리대상자를 분리하고, 문화누리대상자 수요가중치 1.0, 1.2, 1.5를 적용함.
- ML 확률이 있는 중분류는 성별·연령·장애여부별 선호확률을 적용함.
- 음악, 체육용품은 ML 확률이 없어 선호 미반영 SFCA 접근성으로 대체함.
- Huff 선택확률은 공급량과 거리감쇠를 함께 반영해 계산함.
- 거리감쇠는 팀원 구간형, 가우시안 연속형, 감쇠 없음 3개 시나리오를 계산함.

## 주요 산출물
- h3sfca_격자_중분류_접근성.csv
- h3sfca_preference_sensitivity_격자_중분류_접근성.parquet
- h3sfca_preference_sensitivity_중분류_요약.csv
- h3sfca_preference_sensitivity_가맹점_공급수요비.parquet
"""
    div_doc = """# 문화시설 다양성 지수

## 사용 데이터
- 서울 + 외부 25km 문화누리 가맹점
- 대중교통 20분 이내 격자-가맹점 접근 pair
- 서울 100m 격자 기본 테이블

## 전처리/분석 방식
- 서울 격자 기준으로 20분 이내 접근 가능한 문화누리 가맹점의 중분류 종류 수를 계산함.
- diversity_type_n은 실제 접근 가능한 중분류 종류 수임.
- diversity_ratio는 diversity_type_n / 전체 중분류 수로 계산함.
- diversity_vulnerability는 1 - diversity_ratio로 계산함.

## 주요 산출물
- grid_pop_access_diversity.csv
"""
    (DOCS_PATH / "h3sfca_sfca_전처리_사용데이터.txt").write_text(sfca_doc, encoding="utf-8")
    (DOCS_PATH / "h3sfca_ml_전처리_사용데이터.txt").write_text(h3_doc, encoding="utf-8")
    (DOCS_PATH / "diversity_전처리_사용데이터.txt").write_text(div_doc, encoding="utf-8")


def main():
    for path in [
        GRID_COMPETITION_PATH, STORE_COMPETITION_PATH, TRANSIT_PAIR_PATH, SUPPLY_PATH,
        SEOUL_MNC_GRID_PATH, EXTERNAL_MNC_GRID_PATH,
        SEOUL_MNC_AGE_DISABLED_PATH, EXTERNAL_AGE_PATH,
        DISABLED_SIGUNGU_PATH, RESIDENT_POP_PATH, PREFERENCE_PATH
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    grid = load_grid_base()
    store = load_store_supply()
    pair = load_access_pair(store)

    sfca, sfca_facility = run_no_preference_sfca(grid, pair)
    sfca_summary = summarize_access(sfca)
    sfca.to_csv(ACCESS_OUTPUT_PATH / "sfca_no_preference_격자_중분류_접근성.csv", index=False, encoding="utf-8-sig")
    sfca_summary.to_csv(ACCESS_OUTPUT_PATH / "sfca_no_preference_중분류_요약.csv", index=False, encoding="utf-8-sig")
    sfca_facility.to_csv(ACCESS_OUTPUT_PATH / "sfca_no_preference_가맹점_공급수요비.csv", index=False, encoding="utf-8-sig")
    print("\n선호 미반영 SFCA 중분류 요약")
    display(sfca_summary)

    disabled_ratio = parse_disabled_ratio()
    seoul_strata = build_seoul_demand_strata(grid, disabled_ratio)
    external_strata = build_external_demand_strata(disabled_ratio)
    demand_strata = pd.concat([seoul_strata, external_strata], ignore_index=True)
    demand_strata[["일반인구", "문화누리대상자"]] = demand_strata[["일반인구", "문화누리대상자"]].fillna(0).clip(lower=0)
    log(f"선호 적용 수요 strata: {demand_strata.shape}, 일반 {demand_strata['일반인구'].sum():,.0f}, 문화누리 {demand_strata['문화누리대상자'].sum():,.0f}")
    demand = build_preference_demand(grid, demand_strata)
    demand.to_parquet(ACCESS_OUTPUT_PATH / "h3sfca_격자_중분류_선호수요.parquet", index=False)

    scenario_outputs = []
    facility_outputs = []
    for decay_method in DECAY_METHODS:
        for lambda_value in LAMBDA_VALUES:
            log(f"H3SFCA 계산: 거리감쇠={decay_method}, lambda={lambda_value}")
            complete, facility = run_h3sfca_scenario(grid, pair, demand, decay_method, lambda_value)
            scenario_outputs.append(complete)
            facility_outputs.append(facility)
            gc.collect()

    h3_all = pd.concat(scenario_outputs, ignore_index=True)
    h3_all = replace_non_ml_with_sfca(h3_all, sfca)
    h3_facility = pd.concat(facility_outputs, ignore_index=True)
    h3_facility = replace_non_ml_facility_with_sfca(h3_facility, sfca_facility)
    h3_summary = summarize_access(h3_all)

    h3_all.to_parquet(ACCESS_OUTPUT_PATH / "h3sfca_preference_sensitivity_격자_중분류_접근성.parquet", index=False)
    h3_facility.to_parquet(ACCESS_OUTPUT_PATH / "h3sfca_preference_sensitivity_가맹점_공급수요비.parquet", index=False)
    h3_summary.to_csv(ACCESS_OUTPUT_PATH / "h3sfca_preference_sensitivity_중분류_요약.csv", index=False, encoding="utf-8-sig")

    primary = h3_all[
        h3_all["거리감쇠방식"].eq("piecewise")
        & np.isclose(h3_all["문화누리대상자_수요가중치"], PRIMARY_LAMBDA)
    ].copy()
    primary.to_csv(ACCESS_OUTPUT_PATH / "h3sfca_격자_중분류_접근성.csv", index=False, encoding="utf-8-sig")
    primary_summary = summarize_access(primary)
    primary_summary.to_csv(ACCESS_OUTPUT_PATH / "h3sfca_중분류_요약.csv", index=False, encoding="utf-8-sig")

    diversity = build_diversity(grid, pair)
    save_docs()

    run_info = {
        "analysis_scope": "서울 결과 + 외부 25km 경쟁수요/경쟁공급",
        "main_network": "도보 750m 또는 도보 750m 또는 대중교통 20분 이내 pair",
        "primary_lambda": PRIMARY_LAMBDA,
        "lambda_values": LAMBDA_VALUES,
        "decay_methods": DECAY_METHODS,
        "non_ml_categories": NON_ML_CATEGORIES,
        "non_ml_replacement": "선호확률 없는 중분류는 선호 미반영 SFCA 접근성으로 대체",
        "sfca_rows": int(len(sfca)),
        "h3_primary_rows": int(len(primary)),
        "h3_sensitivity_rows": int(len(h3_all)),
        "diversity_rows": int(len(diversity)),
        "seoul_grid_n": int(grid["서울여부"].sum()),
        "outside_grid_n": int((~grid["서울여부"]).sum()),
    }
    (ACCESS_OUTPUT_PATH / "h3sfca_실행정보.json").write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n선호 반영 H3SFCA 기본 시나리오 요약")
    display(primary_summary)
    print("\n민감도 요약 상위")
    display(h3_summary.head(30))
    print("\n문화시설 다양성 요약")
    display(diversity[["diversity_type_n", "reachable_store_n", "diversity_ratio", "diversity_vulnerability"]].describe())
    return {
        "sfca_summary": sfca_summary,
        "h3_primary_summary": primary_summary,
        "h3_sensitivity_summary": h3_summary,
        "diversity": diversity,
        "run_info": run_info,
    }


outputs = main()

## 6. 주요 결과 요약

- 분석 범위: 서울 결과 격자 60,528개, 외부 25km 경쟁 격자 443,058개.
- 네트워크 기준: 차도망 제외, 도보 750m 또는 대중교통 20분 이내 pair 사용.
- 공급량: 팀원 공급량 산정표의 sqrt 보정값 사용, 외부 가맹점은 서울 동일 소분류/중분류 중앙값으로 대체.
- 선호 미반영 SFCA 평균 접근성 상위 중분류: 체육용품, 공연, 체육시설.
- 선호 반영 H3SFCA 기본 시나리오 평균 접근성 상위 중분류: 공연, 미술, 도서.
- ML 선호확률이 없는 음악·체육용품은 선호 미반영 SFCA 접근성으로 대체.
- 산출물의 선호확률_처리방식 칼럼으로 ML선호확률적용 / 선호미반영_SFCA대체를 구분함.
- 문화시설 다양성: 평균 5.12개 중분류, 중앙값 5개 중분류, 최대 10개 중분류.
- 도달 가능 중분류가 0개인 서울 격자: 3,500개.
